# 01 — Data & the Deutsche Bahn Timetables API

This notebook explains how we collect live train data from the DB Timetables API,
what the raw data looks like, and how we parse it into structured trip records.

**Prerequisites:** A valid API key from [developers.deutschebahn.com](https://developers.deutschebahn.com) and a `.env` file in the project root.

## 1. Setup

In [ ]:
import sys
sys.path.append('..')  # import from project root

from scraper import HEADERS, BASE, STATIONS, fetch_plan, fetch_changes, build_changes_map, parse_time

print(f'Tracking {len(STATIONS)} stations')
for name, eva in STATIONS.items():
    print(f'  {name:30s} EVA: {eva}')

## 2. The Two API Endpoints

The DB Timetables API has two endpoints we use:

| Endpoint | URL | What it returns |
|---|---|---|
| **plan** | `/plan/{eva}/{date}/{hour}` | Scheduled timetable for one station, one hour |
| **fchg** | `/fchg/{eva}` | Full changes — actual times, cancellations, platform changes |

We combine them: the **plan** gives us what *should* happen, **fchg** gives us what *actually* happened.

In [ ]:
# Fetch plan + changes for Stuttgart Hbf
eva = STATIONS['Stuttgart Hbf']

plan = fetch_plan(eva)
changes_root = fetch_changes(eva)
changes = build_changes_map(changes_root)

trips = plan.findall('s')
print(f'Plan trips this hour : {len(trips)}')
print(f'Change records       : {len(changes)}')

## 3. Raw XML Structure

Each trip `<s>` element in the plan contains:
- `<tl>` — train label (type, line number)
- `<dp>` — departure info (planned time `pt`, platform `pp`, path `ppth`)
- `<ar>` — arrival info (same fields)

The changes `<s>` element adds:
- `<dp ct=...>` — changed departure time
- `<ar ct=...>` — changed arrival time
- `cs="c"` — cancelled flag

In [ ]:
import xml.etree.ElementTree as ET

# Look at the first trip element
s = trips[0]
print('Trip ID:', s.get('id'))
print('Raw XML:')
print(ET.tostring(s, encoding='unicode'))

## 4. Parsing a Trip

The `ppth` field (planned path) is a `|`-separated list of all stations this train passes through.
We use the **last element** as the train's direction/destination.

In [ ]:
def parse_trip(station_name, eva, s_elem, changes):
    trip_id = s_elem.get('id')
    tl = s_elem.find('tl')
    dp = s_elem.find('dp')
    ar = s_elem.find('ar')
    ref = dp if dp is not None else ar

    train_type = tl.get('c') if tl is not None else None
    line       = ref.get('l') if ref is not None else None
    ppth       = ref.get('ppth', '') if ref is not None else ''
    direction  = ppth.split('|')[-1] if ppth else None
    sched_dep  = parse_time(dp.get('pt')) if dp is not None else None
    sched_arr  = parse_time(ar.get('pt')) if ar is not None else None

    actual_dep = actual_arr = None
    cancelled  = False

    chg = changes.get(trip_id)
    if chg is not None:
        cdp = chg.find('dp')
        car = chg.find('ar')
        if cdp is not None:
            actual_dep = parse_time(cdp.get('ct'))
            cancelled  = cdp.get('cs') == 'c'
        if car is not None:
            actual_arr = parse_time(car.get('ct'))

    return {
        'train_id':     trip_id,
        'train_type':   train_type,
        'line':         line,
        'direction':    direction,
        'station':      station_name,
        'sched_dep':    sched_dep,
        'sched_arr':    sched_arr,
        'actual_dep':   actual_dep,
        'actual_arr':   actual_arr,
        'cancelled':    cancelled,
        'route_stops':  len(ppth.split('|')) if ppth else 0,
    }

# Parse first few trips
for s in trips[:5]:
    trip = parse_trip('Stuttgart Hbf', eva, s, changes)
    delay = None
    if trip['sched_arr'] and trip['actual_arr']:
        delay = (trip['actual_arr'] - trip['sched_arr']).total_seconds() / 60
    print(f"  {trip['train_type']:3s} {str(trip['line']):4s}  →  {str(trip['direction'])[:25]:25s}  "
          f"sched={trip['sched_arr']}  delay={f'{delay:+.0f}min' if delay is not None else 'unknown'}")

## 5. Excluded Train Types

We exclude **IC** and **ICE** trains — they operate on long-distance intercity routes
with very different delay patterns from the S-Bahn/regional trains we focus on.
Including them would add noise without benefit to the local commuter model.

In [ ]:
from collections import Counter

type_counts = Counter()
for s in trips:
    tl = s.find('tl')
    if tl is not None:
        type_counts[tl.get('c')] += 1

print('Train types at Stuttgart Hbf this hour:')
for t, count in type_counts.most_common():
    skipped = ' ← EXCLUDED' if t in ('IC', 'ICE') else ''
    print(f'  {str(t):6s}: {count:3d} trips{skipped}')

## 6. The 60-Second Scrape Loop

The `pipeline.py` script repeats this for all 12 stations every 60 seconds:

```
for each station:
    plan    = fetch_plan(eva)          # scheduled timetable
    changes = fetch_changes(eva)       # real-time updates
    
    for each trip in plan:
        row      = extract fields
        features = build_features(row) # → notebook 02
        pred     = model.predict_one(features)
        delay    = actual_arr - sched_arr
        
        INSERT into trips (upsert)
        INSERT into predictions
        
        if delay is known:
            model.learn_one(features, delay)  # online learning
```

The key insight: **predictions happen before the train arrives, learning happens after.**
The same trip is seen in many consecutive scrape cycles — the DB keeps updating its `fchg` feed
as trains move through the network.